# 06 — Unicidade das atividades: o mesmo treino pelo app e pelo relógio

**Dimensão:** unicidade (ver `AGENTS.md`, seção "Qualidade de dados").

**Objetivo:** contar os treinos que chegaram duas vezes, uma pelo app e outra pelo relógio.

**Pedido:**
1. achar os pares da mesma conta, mesmo tipo e mesma duração, com início a até 5 minutos, um de origem `app` e outro de `garmin` ou `apple_health`;
2. contar os pares por origem;
3. mostrar cinco pares lado a lado.

**Referência (roteiro do professor):** 9.383 pares, 4.184 com o apple_health e 5.199 com o garmin; nos pares, a distância também é igual. Armadilha: o garmin grava o início 3 horas à frente (UTC); sem corrigir o fuso, só aparecem os 4.184 do apple_health.

Nada é removido e nada em `dados/brutos/` é alterado.

## Como o horário de cada origem é tratado

O dicionário diz que `inicio` vem "como veio da origem", e as strings não têm sufixo de fuso: o fuso precisa ser definido por `origem`. A hipótese é: `garmin` em UTC (+3h), `app` e `apple_health` em horário de Brasília. **Antes de corrigir, a hipótese é testada nos próprios dados.**

In [ ]:
import pandas as pd
from pathlib import Path

BRUTOS = Path('../dados/brutos')
atividades = pd.read_parquet(BRUTOS / 'atividades.parquet')
atividades['inicio_ts'] = pd.to_datetime(atividades['inicio'])

CHAVE = ['user_id', 'tipo', 'duracao_min']
app = atividades[atividades['origem'] == 'app']
relogio = atividades[atividades['origem'] != 'app']

In [ ]:
# Teste do fuso: candidatos com mesma conta, tipo e duração, a até 4 horas de distância
candidatos = app.merge(relogio, on=CHAVE, suffixes=('_app', '_relogio'))
candidatos['diferenca_h'] = (candidatos['inicio_ts_relogio'] - candidatos['inicio_ts_app']).dt.total_seconds() / 3600
candidatos = candidatos[candidatos['diferenca_h'].abs() <= 4]
pd.crosstab(candidatos['diferenca_h'].round(0), candidatos['origem_relogio'])

O deslocamento se confirma: os candidatos com o garmin se concentram em +3 h, e com o apple_health em 0 h. Correção: subtrair 3 horas do `inicio` do garmin.

In [ ]:
FUSO_POR_ORIGEM = {'app': 0, 'apple_health': 0, 'garmin': -3}  # horas para chegar ao horário de Brasília

def achar_pares(corrigir_fuso, tolerancia='5min'):
    rel = relogio.copy()
    if corrigir_fuso:
        rel['inicio_ts'] = rel['inicio_ts'] + pd.to_timedelta(rel['origem'].map(FUSO_POR_ORIGEM), unit='h')
    pares = app.merge(rel, on=CHAVE, suffixes=('_app', '_relogio'))
    return pares[(pares['inicio_ts_relogio'] - pares['inicio_ts_app']).abs() <= pd.Timedelta(tolerancia)]

## 1 e 2. Pares por origem

In [ ]:
sem_correcao = achar_pares(corrigir_fuso=False)
pares = achar_pares(corrigir_fuso=True)

resumo = pd.DataFrame({
    'sem corrigir o fuso': sem_correcao['origem_relogio'].value_counts(),
    'com o fuso corrigido': pares['origem_relogio'].value_counts(),
}).fillna(0).astype(int)
resumo.loc['total'] = resumo.sum()
resumo

In [ ]:
assert len(pares) == 9_383
assert pares['origem_relogio'].value_counts().to_dict() == {'garmin': 5_199, 'apple_health': 4_184}
assert len(sem_correcao) == 4_184 and set(sem_correcao['origem_relogio']) == {'apple_health'}
assert not pares['atividade_id_app'].duplicated().any() and not pares['atividade_id_relogio'].duplicated().any(), 'cada treino em no máximo um par'

distancia_igual = (pares['distancia_km_app'].fillna(-1) == pares['distancia_km_relogio'].fillna(-1)).all()
assert distancia_igual
print('Números do roteiro conferidos. Cada treino aparece em no máximo um par; distância igual em todos os pares.')

## 3. Cinco pares lado a lado

In [ ]:
colunas = ['atividade_id', 'origem', 'inicio', 'distancia_km', 'ritmo_min_km', 'fc_media']
lado_a_lado = pares[['user_id', 'tipo', 'duracao_min']
                    + [f'{c}_app' for c in colunas] + [f'{c}_relogio' for c in colunas]]
pd.concat([lado_a_lado[lado_a_lado['origem_relogio'] == 'garmin'].head(3),
           lado_a_lado[lado_a_lado['origem_relogio'] == 'apple_health'].head(2)]).T

## Além do roteiro: a cópia do app e o resíduo em português

No notebook 04, ficou um resíduo de treinos do app em português depois de 15/04/2026, quando o app passou a gravar em inglês. A hipótese era "quem não atualizou o app". Os pares mostram outra explicação.

In [ ]:
PAREADOS_PT = ['corrida', 'caminhada', 'forca', 'pedal']
residuo = app[(app['inicio'] >= '2026-04-15') & app['tipo'].isin(PAREADOS_PT)]
sao_copia = residuo['atividade_id'].isin(pares['atividade_id_app'])
print(f'Treinos do app em português desde 15/04/2026: {len(residuo):,} | cópias de relógio: {sao_copia.sum():,}')
assert sao_copia.all()

fc_vazio_copias = pares['fc_media_app'].isna().mean()
fc_vazio_resto = app.loc[~app['atividade_id'].isin(pares['atividade_id_app']), 'fc_media'].isna().mean()
print(f'fc_media vazio — cópias do app: {fc_vazio_copias:.0%} | demais treinos do app: {fc_vazio_resto:.0%}')

O resíduo em português é **inteiramente** de cópias: o app importa o treino do relógio com o nome original (em português) e com a frequência cardíaca, que o app sozinho quase nunca tem. A hipótese do notebook 04 estava errada e foi corrigida lá.

Consequência prática: o tipo cru e o tipo padronizado dão os mesmos 9.383 pares, porque a cópia do app herda o tipo do relógio.

## Conclusão

- **Roteiro conferido:** 9.383 pares, 5.199 com o garmin e 4.184 com o apple_health; distância igual em todos; sem corrigir o fuso, só os 4.184 do apple_health aparecem.
- **Fuso testado, não suposto:** o garmin fica exatamente 3 h à frente; app e apple_health no mesmo horário.
- **Cada treino aparece em no máximo um par**: a regra não casa um treino com dois.
- **Proposta de deduplicação (não aplicada):** manter o registro do relógio e descartar a cópia do app. O relógio é a fonte original; a cópia não traz nada que ele não tenha.
- **Efeito em contagens:** sem deduplicar, quem usa relógio parece treinar mais. São 9.383 treinos a mais na base, concentrados em quem tem relógio conectado.
- **Correção do notebook 04:** o resíduo em português do app depois de 15/04 são as 4.032 cópias de relógio, não usuários com a versão antiga.

Decisões registradas em `decisoes.md`.